# V2-2 — Doctor-authored report fine-tuning
Private Kaggle notebook. Input: private `lumbar-mri-annotations`; output: adapters and run receipts.
Run smoke first, then full. The target reports may include findings not observable from the eight grading fields; clinical review remains required.


In [ ]:
from pathlib import Path
import subprocess, sys
CODE_SHA = "f8dcd95"
MODE = "smoke"  # change to "full" only after smoke success
RUN_NAME = "v2-2-fold1-smoke-01"  # choose a fresh name for each run
REPO = Path("/kaggle/working/repo")
assert not REPO.exists(), "Start a fresh Save & Run session"
subprocess.run(["git", "clone", "https://github.com/kttt294/MRI-report-generator.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", CODE_SHA], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-kaggle.txt")], check=True)


In [ ]:
import torch
assert torch.cuda.is_available(), "Select GPU T4 in Kaggle Settings"
assert MODE in {"smoke", "full"}
assert RUN_NAME and all(ch.isalnum() or ch in "_-" for ch in RUN_NAME)
print("Mode:", MODE, "GPU:", torch.cuda.get_device_name(0), "Code:", CODE_SHA)


In [ ]:
import subprocess
RUN_DIR = Path("/kaggle/working/runs") / RUN_NAME
subprocess.run([sys.executable, "scripts/run_v2_2_cloud.py",
    "--annotations-root", "/kaggle/input", "--output", str(RUN_DIR),
    "--mode", MODE, "--config", "configs/v2_2_train.yaml",
    "--runtime-minutes", "180" if MODE == "full" else "20"],
    cwd=REPO, check=True)
print("Artifacts:", RUN_DIR)
for path in sorted(RUN_DIR.rglob("train_result.json")):
    print(path.read_text(encoding="utf-8"))
